# 01. 라벨링 도구 (Colab)

`manifest.csv` 우선순위(소리·움직임 많은 순)로 클립을 보여주고 체크박스로 라벨을 달아
`labels.csv` 에 **즉시 저장**합니다. 중단 후 다시 실행하면 미라벨 클립부터 이어집니다.

- 값: 체크=1, 해제=0 — "저장" 버튼을 눌러야 기록됩니다. "건너뜀"은 빈칸 유지.
- D2(움직임)·D8(소리)는 약라벨이 이미 있으므로 **다르게 보일 때만** 수정하면 됩니다.

In [ ]:
# 경로 설정 + Drive 마운트 (Colab)
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path('/content/drive/MyDrive/BabyMon/dataset')   # ← 업로드 위치에 맞게 수정
assert BASE.exists(), f'{BASE} 없음 — Drive 업로드 위치를 확인하세요'

# 클립 폴더 자동 탐색 (clips/ 또는 resized_2/ 또는 BASE 바로 아래)
CLIPS = None
for cand in (BASE/'clips', BASE/'resized_2', BASE):
    if cand.is_dir() and next(cand.glob('*.mp4'), None):
        CLIPS = cand; break
assert CLIPS, f'{BASE} 아래에서 mp4 폴더를 못 찾음 (clips/ 또는 resized_2/)'
assert (BASE/'manifest.csv').exists(), \
    f'{BASE}/manifest.csv 없음 — 로컬 D:\\carved\\dataset\\manifest.csv 와 labels.csv 를 이 폴더로 업로드하세요'
WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)
print('clips:', CLIPS, '/', len(list(CLIPS.glob("*.mp4"))), '개')

In [ ]:
import pandas as pd
LABEL_COLS = ["D1_nose_covered","D2_moving_freq","D3_eyes_open","D4_hands_out",
              "D5_pre_cry","D6_spit_up","D7_crying","D8_baby_sound","D9_mouthing",
              "C1_adult_hand","C2_baby_absent","C3_other_sound"]

man = pd.read_csv(BASE/'manifest.csv')
lab_path = BASE/'labels.csv'
lab = pd.read_csv(lab_path, dtype=str) if lab_path.exists() else pd.DataFrame(columns=['file'])
labeled = set(lab['file'])

# 우선순위: 소리 비율 + 움직임 (정규화 합) 높은 순 — 이벤트성 클립부터
man['prio'] = man['sound_frac'].fillna(0)/man['sound_frac'].max() + \
              pd.to_numeric(man['mean_ydif'], errors='coerce').fillna(0).clip(0,3)/3
queue = man[~man['file'].isin(labeled)].sort_values('prio', ascending=False)['file'].tolist()
print(f'라벨 완료 {len(labeled)} / 대기 {len(queue)}')

In [ ]:
import ipywidgets as W
from IPython.display import display, Video, clear_output
import datetime, csv

idx = 0
out = W.Output()
checks = {c: W.Checkbox(description=c, indent=False) for c in LABEL_COLS}
cry = W.Dropdown(options=['', 'hunger', 'discomfort', 'other'], description='cry_type')

def show():
    with out:
        clear_output()
        if idx >= len(queue):
            print('대기열 끝!'); return
        f = queue[idx]
        row = man[man['file'] == f].iloc[0]
        print(f'[{idx+1}/{len(queue)}] {f}  meanYDIF={row.mean_ydif}  sound={row.sound_frac}')
        display(Video(str(CLIPS/f), embed=True, width=480))
        # 약라벨 미리 채움
        for c in checks.values(): c.value = False
        if str(row.get('weak_D2_moving','')) == '1': checks['D2_moving_freq'].value = True
        if str(row.get('weak_D8_sound','')) == '1': checks['D8_baby_sound'].value = True
        cry.value = ''

def save(_):
    global idx
    f = queue[idx]
    with open(lab_path, 'a', newline='', encoding='utf-8-sig') as fh:
        csv.writer(fh).writerow([f, *[int(checks[c].value) for c in LABEL_COLS],
                                 cry.value, 'me', datetime.datetime.now().isoformat(timespec='seconds')])
    idx += 1; show()

def skip(_):
    global idx; idx += 1; show()

b_save = W.Button(description='저장 → 다음', button_style='success')
b_skip = W.Button(description='건너뜀')
b_save.on_click(save); b_skip.on_click(skip)
display(W.VBox([W.HBox([b_save, b_skip]),
                W.GridBox(list(checks.values()), layout=W.Layout(grid_template_columns='repeat(3, 240px)')),
                cry, out]))
show()

### 팁
- 세션이 끊겨도 `labels.csv` 는 Drive에 남습니다 (같은 파일을 두 번 저장하면 마지막 행이 우선하도록 학습 노트북에서 dedupe 함).
- 특정 라벨만 모으고 싶으면 위 `queue` 정렬을 바꾸세요 (예: `man.sort_values('max_ydif', ascending=False)`).